# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import spacy
import os
from tqdm import tqdm 
from sklearn.metrics import classification_report
import os
from openai import OpenAI
import time
import random

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

# Import Dataset

In [2]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset4.csv')
queries = q_df['question']
q_df['label'] = q_df['label'].str.lower()
q_df['label'] = q_df['label'].replace(mapping)
label = q_df['label'].str.lower().map(label_mapper)
print(q_df['label'].value_counts())

label
synthesis        29
knowledge        22
evaluation       21
comprehension    20
analysis         19
application      15
Name: count, dtype: int64


# Setup API

In [8]:
# Groq

api_key = os.environ.get("GROQ_API_KEY1")

if api_key:
    print('successful')

groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = api_key
)

successful


**Professor**: Knows label defination + counter examples \
**Student**: Knows Context\
**Psycholigist**: knows how to use labels\
**engineer**: Knows context, defination and usage\
**examiner**: does not know context but knows defination

# Professor Persona

## GPT

In [5]:
profg_pred_labels = []

for query in tqdm(queries):
    # Reason
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Think like a university professor and reason about which Bloom's Taxonomy level query belongs to.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Counterexamples:

                    1. “List the steps to apply gradient descent.” → is not Knowledge
                    2. “Explain how you would implement this feature in code.” → is not Comprehension
                    3. “Compare quicksort and mergesort on large datasets.” → is not Application
                    4. “Design an experiment to compare two models.” → is not Analysis
                    5. “Evaluate three proposed architectures and pick one.” → is not Synthesis
                    6. “Describe the algorithm’s steps.” → is not Evaluation
                
                query : {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    chat_completion = groq_client.chat.completions.create(
        messages=[
                {
                    "role": "user",
                    "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY.""",
                }
            ],
            model="openai/gpt-oss-120b",
        )

    reply = chat_completion.choices[0].message.content.lower()

    
    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    profg_pred_labels.append(reply.lower())

100%|██████████| 126/126 [12:13<00:00,  5.82s/it]


In [6]:
print(classification_report(label , [label_mapper[key.lower()] for key in profg_pred_labels]))

              precision    recall  f1-score   support

           0       0.81      0.95      0.88        22
           1       0.68      0.65      0.67        20
           2       0.42      0.33      0.37        15
           3       0.86      0.63      0.73        19
           4       0.70      0.90      0.79        29
           5       0.94      0.81      0.87        21

    accuracy                           0.75       126
   macro avg       0.74      0.71      0.72       126
weighted avg       0.75      0.75      0.74       126



## LLAMA

In [7]:
profl_pred_labels = []

for query in tqdm(queries):
    # Reason
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Think like a university professor and reason about which Bloom's Taxonomy level query belongs to.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Counterexamples:

                    1. “List the steps to apply gradient descent.” → is not Knowledge
                    2. “Explain how you would implement this feature in code.” → is not Comprehension
                    3. “Compare quicksort and mergesort on large datasets.” → is not Application
                    4. “Design an experiment to compare two models.” → is not Analysis
                    5. “Evaluate three proposed architectures and pick one.” → is not Synthesis
                    6. “Describe the algorithm’s steps.” → is not Evaluation
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    chat_completion = groq_client.chat.completions.create(
        messages=[
                {
                    "role": "user",
                    "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY.""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

    reply = chat_completion.choices[0].message.content.lower()

    
    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    profl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [12:18<00:00,  5.86s/it]


In [8]:
print(classification_report(label , [label_mapper[key.lower()] for key in profl_pred_labels]))

              precision    recall  f1-score   support

           0       0.80      0.91      0.85        22
           1       0.61      0.85      0.71        20
           2       0.38      0.20      0.26        15
           3       0.92      0.58      0.71        19
           4       0.70      0.90      0.79        29
           5       0.88      0.67      0.76        21

    accuracy                           0.72       126
   macro avg       0.71      0.68      0.68       126
weighted avg       0.73      0.72      0.71       126



# Student

## GPT

In [9]:
stdg_pred_labels = []

for query in tqdm(queries):
    # Reason
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Think like a student and reason about which Bloom's Taxonomy level query belongs to.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    chat_completion = groq_client.chat.completions.create(
        messages=[
                {
                    "role": "user",
                    "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY.""",
                }
            ],
            model="openai/gpt-oss-120b",
        )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    stdg_pred_labels.append(reply.lower())

100%|██████████| 126/126 [09:55<00:00,  4.73s/it]


In [10]:
print(classification_report(label , [label_mapper[key.lower()] for key in stdg_pred_labels]))

              precision    recall  f1-score   support

           0       0.85      1.00      0.92        22
           1       0.87      0.65      0.74        20
           2       0.67      0.40      0.50        15
           3       0.79      0.58      0.67        19
           4       0.67      1.00      0.81        29
           5       0.89      0.81      0.85        21

    accuracy                           0.78       126
   macro avg       0.79      0.74      0.75       126
weighted avg       0.79      0.78      0.77       126



## LLAMA

In [11]:
stdl_pred_labels = []

for query in tqdm(queries):
    # Reason
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Think like a student and reason about which Bloom's Taxonomy level query belongs to.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    chat_completion = groq_client.chat.completions.create(
        messages=[
                {
                    "role": "user",
                    "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY.""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    stdl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [12:26<00:00,  5.92s/it]


In [12]:
print(classification_report(label , [label_mapper[key.lower()] for key in stdl_pred_labels]))

              precision    recall  f1-score   support

           0       0.86      0.86      0.86        22
           1       0.78      0.70      0.74        20
           2       0.45      0.33      0.38        15
           3       0.80      0.63      0.71        19
           4       0.68      0.93      0.78        29
           5       0.85      0.81      0.83        21

    accuracy                           0.75       126
   macro avg       0.74      0.71      0.72       126
weighted avg       0.75      0.75      0.74       126



# Psychologist

## GPT

In [18]:
psyg_pred_labels = []

for query in tqdm(queries):
    # Reason
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Think like a psychiatrist and reason about which Bloom's Taxonomy level query belongs to.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis

                    Counterexamples:

                    1. “List the steps to apply gradient descent.” → is not Knowledge
                    2. “Explain how you would implement this feature in code.” → is not Comprehension
                    3. “Compare quicksort and mergesort on large datasets.” → is not Application
                    4. “Design an experiment to compare two models.” → is not Analysis
                    5. “Evaluate three proposed architectures and pick one.” → is not Synthesis
                    6. “Describe the algorithm’s steps.” → is not Evaluation

                    Now classify the question into one Bloom’s Taxonomy level:
                
                query : {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    chat_completion = groq_client.chat.completions.create(
        messages=[
                {
                    "role": "user",
                    "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY.""",
                }
            ],
            model="openai/gpt-oss-120b",
        )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    psyg_pred_labels.append(reply.lower())

100%|██████████| 126/126 [11:58<00:00,  5.70s/it]


In [19]:
print(classification_report(label , [label_mapper[key.lower()] for key in psyg_pred_labels]))

              precision    recall  f1-score   support

           0       0.86      0.86      0.86        22
           1       0.78      0.70      0.74        20
           2       0.54      0.47      0.50        15
           3       0.71      0.63      0.67        19
           4       0.68      0.93      0.78        29
           5       1.00      0.76      0.86        21

    accuracy                           0.75       126
   macro avg       0.76      0.73      0.74       126
weighted avg       0.77      0.75      0.75       126



## LLAMA

In [20]:
psyl_pred_labels = []

for query in tqdm(queries):
    # Reason
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Think like a psychiatrist and reason about which Bloom's Taxonomy level query belongs to.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis

                    Counterexamples:

                    1. “List the steps to apply gradient descent.” → is not Knowledge
                    2. “Explain how you would implement this feature in code.” → is not Comprehension
                    3. “Compare quicksort and mergesort on large datasets.” → is not Application
                    4. “Design an experiment to compare two models.” → is not Analysis
                    5. “Evaluate three proposed architectures and pick one.” → is not Synthesis
                    6. “Describe the algorithm’s steps.” → is not Evaluation

                    Now classify the question into one Bloom’s Taxonomy level:
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    chat_completion = groq_client.chat.completions.create(
        messages=[
                {
                    "role": "user",
                    "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY.""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    psyl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [12:46<00:00,  6.08s/it]


In [21]:
print(classification_report(label , [label_mapper[key.lower()] for key in psyl_pred_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.83      0.75      0.79        20
           2       0.64      0.47      0.54        15
           3       0.88      0.74      0.80        19
           4       0.64      0.93      0.76        29
           5       1.00      0.71      0.83        21

    accuracy                           0.79       126
   macro avg       0.81      0.76      0.77       126
weighted avg       0.81      0.79      0.78       126



# Engineer

## GPT

In [4]:
engg_pred_labels = []

for query in tqdm(queries):
    # Reason
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Think like a engineer and reason about which Bloom's Taxonomy level query belongs to.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis

                    Now classify the question into one Bloom’s Taxonomy level:
                
                query : {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    chat_completion = groq_client.chat.completions.create(
        messages=[
                {
                    "role": "user",
                    "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY.""",
                }
            ],
            model="openai/gpt-oss-120b",
        )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    engg_pred_labels.append(reply.lower())

100%|██████████| 126/126 [12:07<00:00,  5.77s/it]


In [5]:
print(classification_report(label , [label_mapper[key.lower()] for key in engg_pred_labels]))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91        22
           1       0.78      0.70      0.74        20
           2       0.46      0.40      0.43        15
           3       0.85      0.58      0.69        19
           4       0.68      0.90      0.78        29
           5       0.90      0.86      0.88        21

    accuracy                           0.76       126
   macro avg       0.76      0.73      0.74       126
weighted avg       0.77      0.76      0.76       126



## LLAMA

In [6]:
engl_pred_labels = []

for query in tqdm(queries):
    # Reason
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Think like a engineer and reason about which Bloom's Taxonomy level query belongs to.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Examples:

                    1. What is the capital of France? → Knowledge
                    2. Use Ohm’s law to calculate the current in a circuit. → Application
                    3. Critique the author’s argument in the article. → Evaluation
                    4. Summarize the main idea of the passage. → Comprehension             
                    5. Examine the causes of World War I. → Analysis                
                    6. Assess the validity of the research study’s conclusions. → Evaluation
                    7. Propose a plan to reduce plastic pollution in cities. → Synthesis
                    8. Define photosynthesis. → Knowledge
                    9. Apply the concept of supply and demand to predict price changes. → Application              
                    10. Create a new ending for the story. → Synthesis
                    11. Explain in your own words what Newton’s First Law means. → Comprehension
                    12. Identify the relationship between exercise and mental health in the study. → Analysis

                    Now classify the question into one Bloom’s Taxonomy level:
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    chat_completion = groq_client.chat.completions.create(
        messages=[
                {
                    "role": "user",
                    "content": f"""
                    Summarize the reasoning into exactly one Bloom's Taxonomy level.
                    Levels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY.""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

    reply = chat_completion.choices[0].message.content.lower()

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    engl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [12:08<00:00,  5.78s/it]


In [7]:
print(classification_report(label , [label_mapper[key.lower()] for key in engl_pred_labels]))

              precision    recall  f1-score   support

           0       0.90      0.86      0.88        22
           1       0.75      0.75      0.75        20
           2       0.54      0.47      0.50        15
           3       0.83      0.79      0.81        19
           4       0.68      0.93      0.78        29
           5       1.00      0.67      0.80        21

    accuracy                           0.77       126
   macro avg       0.78      0.74      0.75       126
weighted avg       0.79      0.77      0.77       126



# Examiner

## GPT

In [9]:
examg_pred_labels = []

for query in tqdm(queries):
    # Reason
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Think like a examiner and reason about which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Now classify the following question:
                
                query : {query}""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Summarize the reasoning into exactly one label from.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY""",
            }
        ],
        model="openai/gpt-oss-120b",
    )

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="openai/gpt-oss-20b",
        )

        reply = chat_completion.choices[0].message.content.lower()

    examg_pred_labels.append(reply.lower())

100%|██████████| 126/126 [10:02<00:00,  4.78s/it]


In [10]:
print(classification_report(label , [label_mapper[key.lower()] for key in examg_pred_labels]))

              precision    recall  f1-score   support

           0       0.87      0.91      0.89        22
           1       0.74      0.70      0.72        20
           2       0.43      0.40      0.41        15
           3       0.86      0.63      0.73        19
           4       0.71      0.93      0.81        29
           5       0.89      0.76      0.82        21

    accuracy                           0.75       126
   macro avg       0.75      0.72      0.73       126
weighted avg       0.76      0.75      0.75       126



## LLAMA

In [11]:
examl_pred_labels = []

for query in tqdm(queries):
    # Reason
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""Think like a examiner and reason about which label it belongs.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Definations:
                    
                    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
                    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
                    3. Application: Using learned information in new concrete situations to solve problems
                    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
                    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
                    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing

                    Now classify the following question:
                
                query : {query}""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    reply = chat_completion.choices[0].message.content.lower()

    # Summarize and Classify
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"""
                    Summarize the reasoning into exactly one label from.
                    Labels: [knowledge, comprehension, application, analysis, synthesis, evaluation]

                    Reasoning: {reply}

                    Answer in one word ONLY""",
            }
        ],
        model="meta-llama/llama-4-scout-17b-16e-instruct",
    )

    while reply not in ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']:
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": f"""Extract answer from previous reponse only in one word without punctuation from: 
                        [knowledge , comprehension , application , analysis, synthesis , evaluation]
                    previous response : {reply}""",
                }
            ],
            model="meta-llama/llama-4-scout-17b-16e-instruct",
        )

        reply = chat_completion.choices[0].message.content.lower()

    examl_pred_labels.append(reply.lower())

100%|██████████| 126/126 [14:19<00:00,  6.82s/it]


In [12]:
print(classification_report(label , [label_mapper[key.lower()] for key in examl_pred_labels]))

              precision    recall  f1-score   support

           0       0.90      0.86      0.88        22
           1       0.75      0.75      0.75        20
           2       0.42      0.33      0.37        15
           3       0.93      0.74      0.82        19
           4       0.65      0.90      0.75        29
           5       0.83      0.71      0.77        21

    accuracy                           0.75       126
   macro avg       0.75      0.72      0.73       126
weighted avg       0.76      0.75      0.74       126



# Save Labels

In [ ]:
label_data = {
    'professor_gpt_cot' : profg_pred_labels,
    'student_gpt_cot' : stdg_pred_labels, 
    'psychiatrist_gpt_cot' : psyg_pred_labels,
    'engineer_gpt_cot' : engg_pred_labels,
    'examiner_gpt' : examg_pred_labels
            }

df = pd.DataFrame(data= label_data)

df.to_csv('cot_persona_gpt.csv', index=False) 

In [ ]:
label_data = {
    'professor_llama_cot' : profl_pred_labels,
    'student_llama_cot' : stdl_pred_labels, 
    'psychiatrist_llama_cot' : psyl_pred_labels,
    'engineer_llama_cot' : engl_pred_labels,
    'examiner_llama' : examl_pred_labels
            }

df = pd.DataFrame(data= label_data)

df.to_csv('cot_persona_llama.csv', index=False) 